# Gradient Descent via API (batch calls)

This notebook demonstrates running gradient descent on the client while calling the API to obtain the loss and analytical gradient at each step using `/evaluate-with-visualization`.

It updates a side-by-side visualization every few iterations:
- Left: feature space (dataset points + decision boundary from current params)
- Right: loss landscape slice over (w1, w2) with the moving optimizer point


In [ ]:
import time
import threading
import asyncio
import requests
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import ipywidgets as widgets

from app.functions import quadratic_binary_loss, quadratic_predict_proba

URL = "http://localhost:8000/evaluate-with-visualization"

In [ ]:
def plot_pair(params, points, y_true, loss_fn, predict_proba_fn, w_idx=(1, 2), grid_radius=2.0, path_ij=None):
    """Plot feature space (left) and loss landscape slice (right).

    If ``path_ij`` is provided (sequence of (w_i, w_j) pairs), we overlay
    the optimization path and a small quiver arrow indicating the latest
    step direction on top of the loss landscape.
    """
    p = np.asarray(params)
    # smaller figure so 2x2 grid fits on screen more comfortably
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(8, 3))
    # left: points + decision boundary
    axl.scatter(points[:, 0], points[:, 1], c=y_true, cmap='coolwarm', s=25, edgecolor='k')
    xs = np.linspace(points[:, 0].min() - 0.5, points[:, 0].max() + 0.5, 200)
    ys = np.linspace(points[:, 1].min() - 0.5, points[:, 1].max() + 0.5, 200)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.column_stack([XX.ravel(), YY.ravel()])
    probs = predict_proba_fn(p, grid)
    ZZ = probs.reshape(XX.shape)
    axl.contour(XX, YY, ZZ, levels=[0.5], colors=['k'])
    axl.set_title('Feature space (points + decision boundary)')
    # right: loss landscape slice
    i, j = w_idx
    center_i, center_j = p[i], p[j]
    # allow grid_radius to be either a scalar or (rad_w1, rad_w2)
    if np.isscalar(grid_radius):
        rad_i = rad_j = float(grid_radius)
    else:
        rad_i, rad_j = map(float, grid_radius)
    grid_i = np.linspace(center_i - rad_i, center_i + rad_i, 120)
    grid_j = np.linspace(center_j - rad_j, center_j + rad_j, 120)
    Gx, Gy = np.meshgrid(grid_i, grid_j)
    losses = np.empty_like(Gx)
    for a in range(Gx.shape[0]):
        for b in range(Gx.shape[1]):
            q = p.copy()
            q[i] = Gx[a, b]
            q[j] = Gy[a, b]
            losses[a, b] = loss_fn(q)
    im = axr.imshow(
        losses,
        extent=(grid_i.min(), grid_i.max(), grid_j.min(), grid_j.max()),
        origin='lower',
        cmap='viridis',
    )
    # draw current point
    axr.scatter([p[i]], [p[j]], color='red', s=50, zorder=3)

    # overlay path and a small arrow for direction of movement
    if path_ij is not None and len(path_ij) > 1:
        path_ij = np.asarray(path_ij)
        # path as connected straight segments
        axr.plot(
            path_ij[:, 0],
            path_ij[:, 1],
            color='white',
            alpha=0.7,
            linewidth=1.0,
            zorder=2,
        )
        # circle markers at each iteration along the path
        axr.scatter(
            path_ij[:, 0],
            path_ij[:, 1],
            s=12,
            facecolors='white',
            edgecolors='black',
            linewidths=0.4,
            zorder=3,
        )
        # last step arrow
        x0, y0 = path_ij[-2]
        x1, y1 = path_ij[-1]
        dx, dy = x1 - x0, y1 - y0
        axr.quiver(
            [x0],
            [y0],
            [dx],
            [dy],
            angles='xy',
            scale_units='xy',
            scale=1.0,
            color='red',
            width=0.003,
            zorder=4,
        )

    axr.set_xlabel(f'w[{i}]')
    axr.set_ylabel(f'w[{j}]')
    axr.set_title('Loss landscape (slice)')
    fig.colorbar(im, ax=axr)
    plt.tight_layout()
    return fig

from app.datasets import get_dataset
Xd, yd = get_dataset('quadratic_binary')

# container for all optimization states
states = []  # each element: {"params": np.ndarray, "loss": float}

# widgets for interactive exploration
# Top row: history review (controlled by slider)
# Bottom row: always shows latest state
epoch_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Step",
    continuous_update=True,  # update as you scrub
)

history_header = widgets.Label("History (step 0 of 0)")
review_loss_label = widgets.Label("History loss: n/a")
latest_header = widgets.Label("Latest (step 0 of 0)")
latest_loss_label = widgets.Label("Latest loss: n/a")

review_out = widgets.Output()
latest_out = widgets.Output()

controls_review = widgets.HBox([history_header, epoch_slider, review_loss_label])
controls_latest = widgets.HBox([latest_header, latest_loss_label])

# two separate UIs: history (top cell) and latest (bottom cell)
history_ui = widgets.VBox([controls_review, review_out])
latest_ui = widgets.VBox([controls_latest, latest_out])


def _plot_state(state, out, loss_label):
    """Helper: draw both panels for a given state into a specific output widget."""
    if state is None:
        return

    params = state["params"]
    loss = state["loss"]

    with out:
        out.clear_output(wait=True)
        fig = plot_pair(
            params,
            Xd,
            yd,
            quadratic_binary_loss,
            quadratic_predict_proba,
            w_idx=(1, 2),
        )
        display(fig)
        plt.close(fig)

    loss_label.value = f"loss: {loss:.4f}"


def update_review(change=None):
    """Update the top row (history) based solely on the slider position."""
    if not states:
        return

    n = len(states)
    idx = min(max(epoch_slider.value, 0), n - 1)
    state = states[idx]
    _plot_state(state, review_out, review_loss_label)
    history_header.value = f"History (step {idx + 1} of {n})"


def update_latest():
    """Update the bottom row to always reflect the latest state."""
    if not states:
        return

    n = len(states)
    idx = n - 1
    state = states[idx]
    _plot_state(state, latest_out, latest_loss_label)
    latest_header.value = f"Latest (step {idx + 1} of {n})"


# Only history row reacts to slider scrubs (if events work in this env)
epoch_slider.observe(update_review, names="value")


# GD loop that calls the API for gradient every iteration
# and feeds the states to the interactive widgets

_running_thread = None  # keep a handle so we don't start multiple runs


def _gd_worker(initial_params, lr=0.5, steps=80, plot_every=5):
    """Background thread that performs GD and updates widgets.

    Running this in a separate thread keeps the main Jupyter event loop
    free so slider scrubs and widget callbacks can be processed normally.
    """
    global _running_thread

    params = np.asarray(initial_params, dtype=float).copy()
    history = [params.copy()]

    # reset recorded states and slider
    states.clear()
    epoch_slider.min = 0
    epoch_slider.max = 0
    epoch_slider.value = 0

    for t in range(steps):
        payload = {"problem_id": "quadratic_binary", "x": params.tolist()}
        try:
            r = requests.post(URL, json=payload, timeout=5.0)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print("API request failed at step", t, e)
            break

        grad = np.asarray(data["gradient"], dtype=float)
        loss = float(data["y"])

        # gradient descent update
        params = params - lr * grad
        history.append(params.copy())

        # record state for interactive visualization
        states.append({"params": params.copy(), "loss": loss})

        # extend slider range as new steps arrive, WITHOUT changing its value
        new_max = len(states) - 1
        if new_max < 0:
            new_max = 0
        epoch_slider.max = new_max

        # keep the history header's "of N" part up to date
        n = len(states)
        current_index = min(max(epoch_slider.value, 0), n - 1)
        history_header.value = f"History (step {current_index + 1} of {n})"

        # bottom row always shows the latest state
        update_latest()

        # initialize the history row once when the first state arrives
        if len(states) == 1:
            update_review()

        # small sleep to avoid hammering the API and UI
        time.sleep(0.05)

    # make sure slider range matches the final history length
    final_max = len(states) - 1 if states else 0
    epoch_slider.max = final_max
    if epoch_slider.value > final_max:
        epoch_slider.value = final_max

    # final refresh
    update_latest()
    update_review()

    _running_thread = None


def gd_via_api(initial_params, lr=0.5, steps=80, plot_every=5):
    """Start the GD run in a background thread.

    This returns immediately so that the notebook UI stays interactive.
    """
    global _running_thread

    if _running_thread is not None and _running_thread.is_alive():
        print("GD is already running.")
        return _running_thread

    thread = threading.Thread(
        target=_gd_worker,
        args=(initial_params, lr, steps, plot_every),
        daemon=True,
    )
    _running_thread = thread
    thread.start()
    return thread


In [ ]:
import numpy as _np


def _plot_state(idx, out, loss_label, header_label, prefix):
    """Draw both panels for the state at a given index with zoom + path.

    - Zoom is adapted to the spread of the path in (w1, w2).
    - A small quiver arrow shows the latest step direction.
    """
    global states

    if not states:
        return

    n = len(states)
    idx = max(0, min(int(idx), n - 1))
    state = states[idx]

    # collect the path in (w1, w2) space up to this index
    i, j = 1, 2
    path_ij = _np.array([s["params"][ [i, j] ] for s in states[: idx + 1]])

    # adaptive zoom: anisotropic radius based on spread of the path, with floors
    center = path_ij[-1]
    max_dev = _np.max(_np.abs(path_ij - center), axis=0)
    # separate radii for w[1] and w[2]
    rad_w1 = max(0.035, float(max_dev[0] * 1.5))  # e.g. ~[-0.10, -0.03]
    rad_w2 = max(0.01, float(max_dev[1] * 1.5))   # e.g. ~[-0.02, 0.00]
    radii = (rad_w1, rad_w2)

    params = state["params"]
    loss = state["loss"]

    with out:
        out.clear_output(wait=True)
        fig = plot_pair(
            params,
            Xd,
            yd,
            quadratic_binary_loss,
            quadratic_predict_proba,
            w_idx=(1, 2),
            grid_radius=radii,
            path_ij=path_ij,
        )
        display(fig)
        plt.close(fig)

    header_label.value = f"{prefix} (step {idx + 1} of {n})"
    loss_label.value = f"loss: {loss:.4f}"


def update_review(change=None):
    if not states:
        return

    n = len(states)
    idx = max(0, min(int(epoch_slider.value), n - 1))
    _plot_state(idx, review_out, review_loss_label, history_header, "History")


def update_latest():
    if not states:
        return

    n = len(states)
    idx = n - 1
    _plot_state(idx, latest_out, latest_loss_label, latest_header, "Latest")


# Clear any old observers (from previous versions) and attach the new one
try:
    epoch_slider.unobserve_all()
except Exception:
    pass
epoch_slider.observe(update_review, names="value")

# Display the history UI in this cell so it has its own output area
display(history_ui)


## Latest cell

In [ ]:
# Run the GD via API example
# This starts the GD loop in a background thread. The history UI
# is displayed in the previous cell; here we show the dedicated
# "Latest" view (header + plots) in its own output area.
display(latest_ui)
init = np.zeros(6)
init[3] = -1.0  # small quadratic bias to start
thread = gd_via_api(init, lr=0.6, steps=20, plot_every=3)


In [ ]:
epoch_slider.value